Hack O Week January week 3

In [ ]:
!pip install flask -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 117544 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.1.1) over (2026.1.1) ...
Setting up cloudflared (2026.1.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify
import re

app = Flask(__name__)

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

synonym_groups = {
    "fees": [
        "fee", "fees", "tuition", "tuition fee", "tuition fees",
        "payment", "pay", "paid", "pay fees", "fee payment",
        "semester fee", "semester fees", "annual fee", "yearly fee",
        "charges", "college charges", "cost", "expense", "pricing",
        "amount", "fee amount", "fee structure", "fee details","value","Tolls","Bills","Levies","Subscription"
    ],
    "payment": [
        "payment", "pay", "paid", "fee payment", "online pay",
        "upi", "card", "debit", "credit", "netbanking",
        "bank transfer", "transaction", "txn", "payment status",
        "pending payment", "payment failed", "payment error",
        "payment receipt", "bill", "invoice", "money"
    ],
    "scholarship": [
        "scholarship", "financial aid", "fee waiver", "concession",
        "grant", "stipend", "education loan help", "govt scholarship",
        "state scholarship", "minority scholarship", "merit scholarship",
        "need based", "discount", "scholarship form", "scholarship portal",
        "scholarship amount", "scholarship status", "renew scholarship"
    ],
    "refund": [
        "refund", "money back", "return money", "fee refund",
        "refund policy", "refund rules", "cancellation refund",
        "withdraw refund", "refund status", "refund process",
        "cancel admission", "withdrawal", "dropout", "drop out",
        "partial refund", "full refund", "50 percent refund"
    ],
    "fine": [
        "fine", "penalty", "late fine", "charges", "extra charges",
        "late payment fine", "library fine", "discipline fine",
        "punishment", "fee due fine", "late fees", "fee penalty",
        "late return penalty", "penalty amount", "fine amount",
        "due fine", "extra fee", "late charges"
    ],

    "admission": [
        "admission", "admissions", "apply", "application",
        "apply now", "enroll", "enrollment", "join",
        "registration", "register", "admission form", "application form",
        "admission process", "procedure", "how to join",
        "selection process", "counselling", "seat allotment"
    ],
    "eligibility": [
        "eligibility", "eligible", "criteria", "requirements",
        "qualification", "minimum marks", "percent required",
        "percentage", "marks criteria", "age limit", "limit",
        "conditions", "who can apply", "eligible for admission",
        "cutoff eligibility", "admission criteria", "academic requirement"
    ],
    "cutoff": [
        "cutoff", "cut off", "minimum score", "minimum marks",
        "rank required", "score required", "marks required",
        "closing rank", "opening rank", "cutoff marks",
        "counselling cutoff", "jee cutoff", "mhtcet cutoff",
        "merit cutoff", "previous year cutoff", "expected cutoff",
        "cutoff list", "cutoff criteria"
    ],
    "entrance": [
        "entrance", "entrance exam", "exam", "test", "aptitude test",
        "sitetee", "siteee", "mhtcet", "jee", "jee mains",
        "cet", "entrance paper", "online test", "exam pattern",
        "syllabus", "test syllabus", "sample paper", "mock test",
        "question paper"
    ],
    "documents": [
        "documents", "document", "doc", "proof", "certificate",
        "marksheet", "id proof", "aadhaar", "aadhar", "photo",
        "passport size", "tc", "transfer certificate",
        "migration", "bonafide", "income certificate",
        "caste certificate", "domicile", "admission documents"
    ],

    "exam": [
        "exam", "exams", "semester exam", "sem exam", "end sem",
        "end semester", "endterm", "mid sem", "midsem",
        "internal", "external", "theory paper", "practical exam",
        "exam schedule", "exam time table", "exam timetable",
        "exam date", "hall ticket", "admit card"
    ],
    "timetable": [
        "timetable", "time table", "schedule", "routine",
        "class timetable", "lecture timetable", "daily timetable",
        "sem timetable", "exam timetable", "exam schedule",
        "calendar", "academic calendar", "timings", "period",
        "lecture time", "lab time", "slot", "slot time",
        "today classes", "tomorrow classes"
    ],
    "backlog": [
        "backlog", "kt", "supplementary", "supply", "repeat exam",
        "arrear", "reappear", "re-exam", "atkt", "carry",
        "back paper", "failed subject", "retest", "retake",
        "improvement exam", "improve marks", "revaluation",
        "compartment", "repeat subject"
    ],
    "result": [
        "result", "results", "marks", "score", "grades",
        "gpa", "cgpa", "sgpa", "percentage", "marklist",
        "marksheet", "grade card", "result date", "when result",
        "download result", "result portal", "result link",
        "pass fail", "performance", "grade"
    ],
    "attendance": [
        "attendance", "present", "absent", "bunk", "bunked",
        "attendance shortage", "attendance criteria", "minimum attendance",
        "75 percent", "attendance percentage", "attendance policy",
        "shortage", "condonation", "attendance fine", "attendance warning",
        "attendance requirement", "lecture attendance", "lab attendance"
    ],
    "assignment": [
        "assignment", "homework", "task", "project work",
        "submission", "submit", "deadline", "due date",
        "file upload", "upload", "report", "lab report",
        "seminar", "ppt", "presentation", "classwork",
        "practice", "worksheet", "evaluation", "internal marks"
    ],

    "erp": [
        "erp", "portal", "student portal", "college portal",
        "login", "sign in", "signin", "dashboard", "website portal",
        "erp login", "username", "password", "forgot password",
        "reset password", "account", "profile", "update details",
        "student account", "erp issue", "portal issue"
    ],
    "password": [
        "password", "pass", "forgot password", "reset password",
        "change password", "login password", "otp", "verification",
        "account locked", "locked", "invalid password", "wrong password",
        "username problem", "cannot login", "login error", "security question",
        "update password", "new password"
    ],

    "hostel": [
        "hostel", "hostels", "room", "accommodation", "stay",
        "residence", "boarding", "lodging", "in campus",
        "boys hostel", "girls hostel", "mess", "canteen food",
        "hostel fees", "hostel allotment", "warden",
        "hostel rules", "curfew", "in time", "out time"
    ],
    "mess": [
        "mess", "canteen", "food", "meal", "meals", "breakfast",
        "lunch", "dinner", "snacks", "mess fee", "food charges",
        "menu", "mess menu", "mess timing", "canteen timing",
        "food quality", "mess complaint", "veg", "non veg",
        "mess card", "meal card"
    ],
    "transport": [
        "transport", "bus", "college bus", "bus facility",
        "pickup", "drop", "route", "bus route", "timing bus",
        "transport fee", "bus fees", "travel", "commute",
        "shuttle", "van", "auto", "cab", "ola", "uber",
        "nearest station", "railway station", "airport"
    ],

    "library": [
        "library", "books", "borrow", "issue book", "return book",
        "renew book", "library card", "study room", "reading room",
        "ebook", "digital library", "journal", "magazine",
        "reference", "book bank", "late fine", "library hours",
        "library timing", "silent zone"
    ],
    "wifi": [
        "wifi", "wi fi", "wi-fi", "internet", "network", "lan",
        "college wifi", "campus wifi", "password wifi",
        "wifi login", "slow internet", "no internet",
        "connection", "connected but no internet", "router",
        "hotspot", "data", "speed", "network issue",
        "wifi issue", "internet problem"
    ],
    "canteen": [
        "canteen", "cafeteria", "food court", "snacks",
        "tea", "coffee", "juice", "lunch", "breakfast",
        "dinner", "menu", "prices", "canteen timing",
        "canteen time", "canteen open", "canteen close",
        "food quality", "canteen complaint", "mess"
    ],

    "placement": [
        "placement", "placements", "job", "jobs", "job offer",
        "company", "companies", "campus drive", "placement drive",
        "recruitment", "package", "ctc", "salary", "offer letter",
        "placement cell", "tpo", "training and placement",
        "career", "career support", "placement record", "top recruiters"
    ],
    "internship": [
        "internship", "intern", "internships", "training",
        "industrial training", "summer internship", "winter internship",
        "stipend", "internship offer", "internship letter",
        "internship certificate", "project internship",
        "mandatory internship", "internship form", "internship approval",
        "internship report", "internship duration", "company internship"
    ],

    "contact": [
        "contact", "phone", "call", "mobile", "number",
        "email", "mail", "helpline", "helpdesk", "support",
        "query", "enquiry", "inquiry", "office contact",
        "reception", "admission office", "admin office",
        "whatsapp", "connect", "reach out", "website"
    ],
    "location": [
        "location", "address", "where", "where is college",
        "how to reach", "map", "google map", "direction",
        "nagpur campus", "wathoda", "area", "near", "nearby",
        "nearest landmark", "pin code", "pincode",
        "route", "travel", "distance", "campus address"
    ],
}

faq_intents = [
    {
        "intent": "fees",
        "groups": ["fees", "payment", "scholarship", "fine"],
        "reply": " <b>Fees:</b><br>₹1,35,000 per semester<br> 5% increment each year (as per policy)"
    },
    {
        "intent": "admission",
        "groups": ["admission", "eligibility", "cutoff", "entrance", "documents"],
        "reply": " <b>Admissions:</b><br> Through MHTCET / SITEEE / JEE<br> Selection is merit-based."
    },
    {
        "intent": "academics",
        "groups": ["exam", "timetable", "backlog", "result", "attendance", "assignment"],
        "reply": " <b>Academics:</b><br>Ask me about exam timetable, results, attendance policy, backlogs & assignments."
    },
    {
        "intent": "hostel",
        "groups": ["hostel", "mess", "transport"],
        "reply": " <b>Hostel & Campus Stay:</b><br> Boys & Girls Hostel Available<br> Mess facility included"
    },
    {
        "intent": "campus",
        "groups": ["library", "wifi", "canteen", "location"],
        "reply": " <b>Campus Facilities:</b><br>Library, WiFi and Canteen facilities are available for students."
    },
    {
        "intent": "placements",
        "groups": ["placement", "internship"],
        "reply": " <b>Placements & Internships:</b><br>We have a dedicated placement cell supporting internships and campus recruitment drives."
    },
    {
        "intent": "contact",
        "groups": ["contact"],
        "reply": " <b>Contact:</b><br>Phone: 07126192370<br>Email: btechadmissionsnagpur@siu.edu.in"
    },
    {
        "intent": "refund",
        "groups": ["refund"],
        "reply": " <b>Refund Policy:</b><br> Full refund before academic year starts<br> 50% refund till first semester (as per rules)"
    },
]

# -------------------- MATCHING --------------------
def get_bot_reply(user_message: str):
    msg = normalize(user_message)

    for intent in faq_intents:
        for grp in intent["groups"]:
            for kw in synonym_groups.get(grp, []):
                if kw in msg:
                    return intent["reply"]

    return (
        " Sorry, I didn’t understand that.<br>"
        "Try asking about:<br>"
        " Fees / Payment<br>"
        " Admissions<br>"
        " Exam / Result / Timetable<br>"
        " Hostel / Mess<br>"
        " Library / WiFi / Canteen<br>"
        " Placements / Internship<br>"
        " Contact"
    )

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/chat", methods=["POST"])
def chat():
    user_message = request.json.get("message", "")
    reply = get_bot_reply(user_message)
    return jsonify({"reply": reply})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


Overwriting app.py


In [ ]:
!mkdir -p templates

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Symbiosis Institute of Technology, Nagpur</title>

  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">

  <style>
    :root{
      --primary:#c62828;
      --text:#1f2937;
      --bg:#0b1220;
      --shadow: 0 20px 60px rgba(0,0,0,0.35);
      --radius: 18px;
    }
    *{box-sizing:border-box; margin:0; padding:0;}
    body{
      font-family:'Poppins',sans-serif;
      color:white;
      background:
        radial-gradient(1000px 400px at 20% 20%, rgba(198,40,40,0.25), transparent 60%),
        radial-gradient(900px 500px at 80% 10%, rgba(255,255,255,0.12), transparent 60%),
        linear-gradient(160deg, #070b13, #0b1220, #070b13);
      overflow-x:hidden;
    }
    a{text-decoration:none; color:inherit;}
    html{scroll-behavior:smooth;}

    nav{
      position:sticky;
      top:0;
      z-index:999;
      backdrop-filter: blur(14px);
      background: rgba(0,0,0,0.35);
      border-bottom:1px solid rgba(255,255,255,0.08);
    }
    .nav-wrap{
      max-width:1100px;
      margin:auto;
      padding:14px 18px;
      display:flex;
      justify-content:space-between;
      align-items:center;
    }
    .brand{
      display:flex;
      align-items:center;
      gap:12px;
      font-weight:700;
    }
    .brand-badge{
      width:40px;
      height:40px;
      border-radius:12px;
      display:grid;
      place-items:center;
      background: linear-gradient(135deg, var(--primary), #ff5252);
      box-shadow:0 10px 30px rgba(198,40,40,0.35);
      font-size:18px;
    }
    .nav-links{display:flex; gap:18px; font-size:14px; opacity:0.9;}
    .nav-links a{padding:8px 12px; border-radius:12px; transition:0.2s;}
    .nav-links a:hover{background: rgba(255,255,255,0.08);}

    header{
      max-width:1100px;
      margin:auto;
      padding:70px 18px 30px 18px;
      display:grid;
      grid-template-columns: 1.2fr 0.8fr;
      gap:26px;
      align-items:center;
    }
    .hero h1{font-size:42px; line-height:1.15; margin-bottom:14px;}
    .hero p{opacity:0.8; font-size:15px; line-height:1.7; margin-bottom:20px;}
    .cta{display:flex; gap:12px; flex-wrap:wrap;}
    .btn{
      border:none; cursor:pointer;
      padding:12px 16px;
      border-radius:14px;
      font-weight:600;
      font-size:14px;
      transition:0.2s;
      display:inline-flex;
      gap:10px;
      align-items:center;
    }
    .btn-primary{
      background: linear-gradient(135deg, var(--primary), #ff5252);
      box-shadow:0 14px 40px rgba(198,40,40,0.35);
      color:white;
    }
    .btn-primary:hover{transform: translateY(-2px);}
    .btn-secondary{
      background: rgba(255,255,255,0.10);
      border:1px solid rgba(255,255,255,0.12);
    }

    .container{max-width:1100px; margin:auto; padding:30px 18px 80px 18px;}
    .grid{display:grid; grid-template-columns: repeat(2, 1fr); gap:18px;}
    .card{
      background: rgba(255,255,255,0.10);
      border:1px solid rgba(255,255,255,0.12);
      border-radius: var(--radius);
      padding:20px;
      box-shadow:0 12px 40px rgba(0,0,0,0.22);
      transition:0.2s;
    }
    .card:hover{transform: translateY(-4px);}

    footer{
      padding:18px;
      text-align:center;
      opacity:0.6;
      border-top:1px solid rgba(255,255,255,0.08);
      background: rgba(0,0,0,0.35);
      backdrop-filter: blur(12px);
    }

    #chatbot-icon{
      position:fixed;
      bottom:22px; right:22px;
      width:64px; height:64px;
      border-radius:20px;
      background: linear-gradient(135deg, var(--primary), #ff5252);
      display:grid; place-items:center;
      font-size:30px;
      cursor:pointer;
      box-shadow:0 14px 40px rgba(198,40,40,0.45);
      transition:0.2s;
      user-select:none;
      z-index:999;
    }

    @keyframes dance{
      0% { transform: rotate(0deg); }
      25% { transform: rotate(-16deg); }
      50% { transform: rotate(16deg); }
      75% { transform: rotate(-12deg); }
      100% { transform: rotate(0deg); }
    }
    .dance{animation:dance 0.5s ease-in-out;}

    #chatbot{
      display:none;
      position:fixed;
      bottom:100px; right:22px;
      width:340px; max-width:90vw;
      border-radius:22px;
      overflow:hidden;
      border:1px solid rgba(255,255,255,0.14);
      background: rgba(15, 18, 30, 0.88);
      backdrop-filter: blur(18px);
      box-shadow:0 18px 70px rgba(0,0,0,0.55);
      z-index:999;
    }
    #chatbot-header{
      background: linear-gradient(135deg, var(--primary), #ff5252);
      padding:12px 14px;
      display:flex;
      justify-content:space-between;
      align-items:center;
      font-weight:700;
    }
    #close-bot{
      cursor:pointer;
      padding:6px 10px;
      border-radius:12px;
      font-weight:700;
      background: rgba(255,255,255,0.18);
      border:1px solid rgba(255,255,255,0.18);
    }
    #chatbot-body{height:280px; overflow-y:auto; padding:14px; font-size:14px;}
    .msg{margin-bottom:12px; display:flex;}
    .msg.you{justify-content:flex-end;}
    .bubble{max-width:80%; padding:10px 12px; border-radius:18px; line-height:1.5;}
    .bubble.bot{background: rgba(255,255,255,0.08); border-top-left-radius:6px;}
    .bubble.you{background: linear-gradient(135deg, rgba(198,40,40,0.95), rgba(255,82,82,0.95)); border-top-right-radius:6px;}

    #chatbot-input{display:flex; gap:10px; padding:12px; border-top:1px solid rgba(255,255,255,0.10);}
    #userInput{
      flex:1;
      border:none; outline:none;
      background: rgba(255,255,255,0.08);
      border:1px solid rgba(255,255,255,0.12);
      color:white;
      border-radius:14px;
      padding:12px 12px;
      font-size:14px;
    }
    #sendBtn{
      border:none; cursor:pointer;
      padding:12px 14px;
      border-radius:14px;
      font-weight:700;
      background: linear-gradient(135deg, var(--primary), #ff5252);
      color:white;
      box-shadow:0 12px 30px rgba(198,40,40,0.35);
      transition:0.2s;
    }

    @media(max-width: 900px){
      header{grid-template-columns:1fr; padding-top:44px;}
      .hero h1{font-size:34px;}
      .grid{grid-template-columns: 1fr;}
    }
  </style>
</head>

<body>
  <nav>
    <div class="nav-wrap">
      <div class="brand">
        <div class="brand-badge">SIT</div>
        <div>
          <div style="font-size:14px; opacity:0.85;">Symbiosis Institute of Technology</div>
          <div style="font-size:12px; opacity:0.65;">Nagpur Campus</div>
        </div>
      </div>

      <div class="nav-links">
        <a href="#about">About</a>
        <a href="#admission">Admissions</a>
        <a href="#faculty">Faculty</a>
        <a href="#alumni">Alumni</a>
        <a href="#contact">Contact</a>
      </div>
    </div>
  </nav>

  <header>
    <div class="hero">
      <h1>Modern Engineering Education. <span style="color:#ff6b6b;">Innovation</span> in Every Step.</h1>
      <p>
        Welcome to <b>Symbiosis Institute of Technology, Nagpur</b>. Explore academics, admissions, campus life,
        and interact with our smart FAQ chatbot for quick answers.
      </p>
      <div class="cta">
        <a href="#admission" class="btn btn-primary"> Explore Admissions</a>
        <a href="#contact" class="btn btn-secondary"> Contact Office</a>
      </div>
    </div>

    <div class="card">
      <h2>Quick Highlights</h2>
      <p> Top Faculty |  Career Focus |  Innovation |  Smart HelpBot</p>
    </div>
  </header>

  <div class="container">
    <div class="grid">
      <section id="about" class="card">
        <h2> About Us</h2>
        <p>SIT Nagpur focuses on excellence in engineering education, research, and innovation.</p>
      </section>

      <section id="admission" class="card">
        <h2> Admissions</h2>
        <p>Admissions are through MHTCET, SITEEE and JEE with merit-based selection.</p>
      </section>

      <section id="faculty" class="card">
        <h2> Faculty</h2>
        <p>Highly qualified faculty providing mentoring and career guidance.</p>
      </section>

      <section id="alumni" class="card">
        <h2> Alumni</h2>
        <p>Strong alumni network supporting internships, mentoring and placements.</p>
      </section>

      <section id="contact" class="card" style="grid-column:1/-1;">
        <h2> Contact Us</h2>
        <p><b>Address:</b> Wathoda, Nagpur - 440008</p>
        <p><b>Phone:</b> 07126192370</p>
        <p><b>Email:</b> btechadmissionsnagpur@siu.edu.in</p>
      </section>
    </div>
  </div>

  <div id="chatbot-icon" onclick="robotClick()"></div>

  <div id="chatbot">
    <div id="chatbot-header">
      SIT HelpBot
      <span id="close-bot" onclick="toggleChatbot()">✖</span>
    </div>

    <div id="chatbot-body">
      <div class="msg">
        <div class="bubble bot">Hello!  Ask me about fees, admissions, exams, hostel, library, placements etc.</div>
      </div>
    </div>

    <div id="chatbot-input">
      <input type="text" id="userInput" placeholder="Type your question..." />
      <button id="sendBtn" onclick="sendMessage()">Send</button>
    </div>
  </div>

  <footer>© 2026 Symbiosis Institute of Technology, Nagpur</footer>

  <script>
    function toggleChatbot() {
      const bot = document.getElementById("chatbot");
      bot.style.display = (bot.style.display === "block") ? "none" : "block";
    }

    function robotClick() {
      const icon = document.getElementById("chatbot-icon");
      icon.classList.add("dance");
      setTimeout(() => {
        icon.classList.remove("dance");
        toggleChatbot();
      }, 500);
    }

    function addMessage(text, who="bot") {
      const body = document.getElementById("chatbot-body");
      const wrap = document.createElement("div");
      wrap.className = "msg " + (who === "you" ? "you" : "");
      const bubble = document.createElement("div");
      bubble.className = "bubble " + (who === "you" ? "you" : "bot");
      bubble.innerHTML = text;
      wrap.appendChild(bubble);
      body.appendChild(wrap);
      body.scrollTop = body.scrollHeight;
    }

    async function sendMessage() {
      const input = document.getElementById("userInput");
      const msg = input.value.trim();
      if (!msg) return;

      addMessage(msg, "you");
      input.value = "";
      addMessage("<i>Typing...</i>", "bot");

      let response = await fetch("/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ message: msg })
      });

      let data = await response.json();
      const body = document.getElementById("chatbot-body");
      body.removeChild(body.lastChild);
      addMessage(data.reply, "bot");
    }

    document.getElementById("userInput").addEventListener("keypress", function(e){
      if(e.key === "Enter") sendMessage();
    });
  </script>
</body>
</html>


Overwriting templates/index.html


In [ ]:
!pkill -f app.py
!python app.py &>/content/flask_log.txt &

In [ ]:
!tail -n 20 /content/flask_log.txt

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit


In [ ]:
!pkill -f cloudflared
!cloudflared tunnel --url http://127.0.0.1:5000 --no-autoupdate > tunnel.txt 2>&1 &



In [ ]:
import time
time.sleep(8)
!grep -o "https://[-a-z0-9]*\.trycloudflare.com" tunnel.txt | head -n 1

https://symposium-hawk-ground-spyware.trycloudflare.com
